# 05 — LangGraph Agent

## About

**Purpose:** Build a LangGraph agent that routes user questions to either the RAG pipeline (Q&A over LEIE exclusions) or the XGBoost risk-scoring model based on keyword detection.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-16<br>
**Notes:** Loads artifacts produced by notebooks 03 (XGBoost model via MLflow) and 04 (FAISS index). Gemini API is called inside the RAG tool — re-running the tests in section 8 hits the API. Routing is keyword-based Python (not LLM-based).<br>
**Description:** Loads the FAISS vector store from 04 and the XGBoost model logged by 03 via MLflow. Defines two tools: `query_leie_rag` (FAISS retriever + Gemini answer) and `score_provider_risk` (XGBoost prediction on a hardcoded feature row — POC stub). Builds a `StateGraph` with a router node (keyword detection on the question) and a tool executor node. Compiles and tests both routing paths end-to-end.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-05-16 | 1.0     | Ganapathy K | Initial version |

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [ ]:
import warnings
import os
import logging
from datetime import datetime
from pathlib import Path
from typing import TypedDict

import pandas as pd
from dotenv import load_dotenv

from google import genai

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

import mlflow.xgboost

warnings.filterwarnings("ignore")

### 1.2 Configure logging

In [ ]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [ ]:
faiss_index_path = Path("../data/processed/faiss_leie_index")
mlflow_model_path = "../notebooks/mlruns/0/models/m-787607a155fb46b29bc9fcd13f990822/artifacts"
mlflow_tracking_uri = Path("../notebooks/mlruns").resolve().as_uri()

embedding_model_name = "all-MiniLM-L6-v2"
gemini_model_name = "gemini-2.5-flash"
retriever_k = 3

google_api_key_env_var = "GOOGLE_API_KEY_HEALTHCARE_PROVIDER_TERMINATION" 

### 1.4 Authentication

In [ ]:
load_dotenv()
google_api_key = os.getenv(google_api_key_env_var)
client = genai.Client(api_key=google_api_key)
logger.info(f"Gemini client ready — API key loaded: {google_api_key is not None}")

## 2. Load FAISS Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

vector_store = FAISS.load_local(faiss_index_path, embeddings, allow_dangerous_deserialization=True)

logger.info(f"FAISS index loaded — {vector_store.index.ntotal} vectors")

## 3. Load XGBoost Model

In [ ]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
model = mlflow.xgboost.load_model(mlflow_model_path)
logger.info(f"XGBoost model loaded from: {mlflow_model_path}")

## 4. Define Tools

In [ ]:
def query_leie_rag(question: str) -> str:
    retriever = vector_store.as_retriever(search_kwargs={"k": retriever_k})
    retrieved_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"Based on the following records:\n{context}\n\nAnswer this question: {question}"
    response = client.models.generate_content(model=gemini_model_name, contents=prompt)
    return response.text


def score_provider_risk(npi: int) -> str:
    sample = pd.DataFrame([{
        "Entity Type Code": 1,
        "Provider Business Mailing Address State Name": 17,
        "Provider Business Mailing Address Telephone Number": 5551234567,
        "Provider Business Practice Location Address State Name": 17,
        "Healthcare Provider Taxonomy Code_1": 42,
        "Provider License Number State Code_1": 17,
        "Provider Enumeration Year": 2010,
        "Last Update Year": 2022,
        "Provider Sex Code_F": 0,
        "Provider Sex Code_M": 1,
        "Provider Sex Code_U": 0,
        "Healthcare Provider Primary Taxonomy Switch_1_N": 0,
        "Healthcare Provider Primary Taxonomy Switch_1_Y": 1,
        "Is Sole Proprietor_N": 1,
        "Is Sole Proprietor_X": 0,
        "Is Sole Proprietor_Y": 0
    }])
    prediction = model.predict(sample)
    score = float(prediction[0])
    return f"Risk score for NPI {npi}: {score:.4f} (higher = more risk)"


logger.info("Tools defined: query_leie_rag, score_provider_risk")

## 5. Define Agent State

In [ ]:
class AgentState(TypedDict):
    question: str
    tool_used: str
    answer: str


logger.info("Agent state ready")

## 6. Define Graph Nodes

In [ ]:
def router_node(state: AgentState) -> AgentState:
    question = state["question"].lower()
    if "risk" in question or "score" in question or "npi" in question:
        state["tool_used"] = "score_provider_risk"
    else:
        state["tool_used"] = "query_leie_rag"
    return state


def tool_node(state: AgentState) -> AgentState:
    tool_name = state["tool_used"]
    question = state["question"]
    if tool_name == "score_provider_risk":
        state["answer"] = score_provider_risk(1234567890)
    else:
        state["answer"] = query_leie_rag(question)
    return state


logger.info("Nodes defined: router_node, tool_node")

## 7. Build and Compile Graph

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("router", router_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("router")
graph.add_edge("router", "tool")
graph.add_edge("tool", END)

agent = graph.compile()
logger.info("Agent compiled successfully")

## 8. Test
### 8.1 RAG path

In [ ]:
result = agent.invoke({"question": "Are there any excluded providers in Texas?", "tool_used": "", "answer": ""})
print("Tool used:", result["tool_used"])
print("Answer:", result["answer"])

### 8.2 Risk scoring path

In [ ]:
result_2 = agent.invoke({"question": "What is the risk score for NPI 1234567890?", "tool_used": "", "answer": ""})
print("Tool used:", result_2["tool_used"])
print("Answer:", result_2["answer"])

## 9. Summary

**Pitch:** Built a LangGraph agent that routes user questions to one of two tools — a RAG pipeline for exclusion Q&A or an XGBoost classifier for risk scoring — using keyword-based routing.

### Approach
- Two tools: `query_leie_rag` (FAISS + Gemini, top-3 retrieval) and `score_provider_risk` (XGBoost prediction on a hardcoded feature row — POC stub, not a real NPI lookup)
- `StateGraph` with two nodes: `router` (keyword detection: risk/score/npi → model, else → RAG) and `tool` (executes the chosen tool)
- Both routing paths verified end-to-end

### Tested
- "Are there any excluded providers in Texas?" → routed to `query_leie_rag`
- "What is the risk score for NPI 1234567890?" → routed to `score_provider_risk`

### Limitations
- `score_provider_risk` ignores the NPI argument and uses a hardcoded feature row — needs a real NPI → feature lookup for production
- Router is keyword-based Python, not LLM-based; will miss paraphrased questions

### Next iteration
- Replace `FAISS.load_local` with a Qdrant client connection — same self-hosted-in-GCP-project reasoning as notebook 04
- Add Langfuse tracing across both tool paths (Gap 2 from the AI Engineer career research)